---
Scenario C: Manufacturing Quality (Pandas)

data setup
creating the dataframes for batches, quality tests and deviations

In [ ]:
import pandas as pd

# we are seting up the 3 dataframes for this senario
batches = pd.DataFrame({
    'batch_id': ['B1', 'B2', 'B3'],
    'product': ['DrugA', 'DrugA', 'DrugB'],
    'facility': ['Plant1', 'Plant2', 'Plant1']
})

quality_tests = pd.DataFrame({
    'test_id': ['T1', 'T2', 'T3'],
    'batch_id': ['B1', 'B1', 'B2'],
    'test_type': ['purity', 'stability', 'purity'],
    'result': ['pass', 'fail', 'pass']
})

deviations = pd.DataFrame({
    'deviation_id': ['D1'],
    'batch_id': ['B1'],
    'description': ['temperature excursion']
})

print('batches:'); print(batches)
print('\nquality_tests:'); print(quality_tests)
print('\ndeviations:'); print(deviations)

Q1: Show all batches and their quality test results
- "join used" inner merge
- i used inner merge becuase the question says "only batches with tests".
- inner merge only keeps rows that match in both dataframes, so B3 gets droped becuase it has no tests
- i assumed every test has a valid batch_id.

In [ ]:
# Q1: inner merge - only batches that have test results
# B3 has no tests so it gets droped
c_q1 = batches.merge(quality_tests, on='batch_id', how='inner')[['batch_id', 'test_type', 'result']]
c_q1

Q2: Show all batches, including those without tests
- join used: left merge
- i used left merge becuase we need all batches even no test
- batches is the left dataframe so all 3 batches stay in the result.
- B3 has no tests so test_type and result show as null.


In [ ]:

# B3 shows up with null becuase it has no test records
c_q2 = batches.merge(quality_tests, on='batch_id', how='left')[['batch_id', 'test_type', 'result']]
c_q2

Q3: Find batches with both failed tests and deviations
- join used: inner merge between two filterd sets
- i need batches that have both conditions: at least one faild test and at least one deviation.
- so i first filter quality_tests for fails, get the unique batch_ids. then get unique batch_ids from deviation.
- then inner merge gived me only the batch_ids that apear in both list
- i assumed a batch needs at least one fail and at least one deviation to qualify

In [ ]:
# Q3: inner merge of two filterd sets - batches with faild tests and deviations

failed_batches = quality_tests[quality_tests['result'] == 'fail'][['batch_id']].drop_duplicates()
deviation_batches = deviations[['batch_id']].drop_duplicates()
c_q3 = failed_batches.merge(deviation_batches, on='batch_id', how='inner')
c_q3

Q4: Show batch-level counts of tests and deviations
- join used:left merge (two times) with groupby aggregation
- i want to count how many tests and deviations each batch has so first i groupby to get counts per batch_id in each dataframe and then i left merge both count  dataframes onto batches
- i used left merge becuase some batches may have 0 tests or 0 deviations and i dont want to lose them

In [ ]:
# Q4: double left merge with aggregation - count tests and deviations per batch
# groupby first to get counts, then left merge onto batches so no batch is lost
test_counts = quality_tests.groupby('batch_id').size().reset_index(name='test_count')
dev_counts = deviations.groupby('batch_id').size().reset_index(name='deviation_count')
c_q4 = batches[['batch_id']].merge(test_counts, on='batch_id', how='left').merge(dev_counts, on='batch_id', how='left')
c_q4['test_count'] = c_q4['test_count'].fillna(0).astype(int)
c_q4['deviation_count'] = c_q4['deviation_count'].fillna(0).astype(int)
c_q4

Q5: Find batches with no deviations
- join used: left merge + filter nan "anti-join" 
- same anti-join idea i used before. left megre batches to deviation, then keep only rows where deviation_id is null.
- a batch with zero deviation records is deviation-free.
- B2 and B3 have no deviation.

In [ ]:
# Q5: anti-join,  batchs with no deviations
# left merge then filtr where deviation_id is Nan that is no maching
merged = batches.merge(deviations, on='batch_id', how='left')
c_q5 = merged[merged['deviation_id'].isna()][['batch_id']]
c_q5

---
Scenario D: Marketing Campaign Performance (Pandas)

data setup is creating the dataframes for customers, campaign sends, and clicks.

In [ ]:
import pandas as pd

# it is seting up the 3 dataframes for this senario
customers_d = pd.DataFrame({
    'customer_id': [1, 2, 3],
    'email': ['a@email.com', 'b@email.com', 'c@email.com']
})

campaign_sends = pd.DataFrame({
    'send_id': ['S1', 'S2', 'S3'],
    'customer_id': [1, 2, 1],
    'campaign_id': ['C1', 'C1', 'C2'],
    'send_date': ['2024-01-01', '2024-01-01', '2024-02-01']
})

clicks = pd.DataFrame({
    'click_id': ['CL1'],
    'send_id': ['S1'],
    'click_date': ['2024-01-02']
})

print('customers:'); print(customers_d)
print('\ncampaign_sends:'); print(campaign_sends)
print('\nclicks:'); print(clicks)

Q1: Show all campaign sends with customer emails
- join used:inner merge 
- i only want sends that belong to a real customer so inner merge works
- i assumed every customer_id in campaign_sends is valid.

In [ ]:
# Q1: inner merge - sends with customer emails
# only send with known customres show up
d_q1 = customers_d.merge(campaign_sends, on='customer_id', how='inner')[['email', 'campaign_id', 'send_date']]
d_q1

Q2: Identify whether each campaign send resulted in a click
- join used: left merge 
- i need all sends to show up, so sends is the left dataframe.
- left merge with clicks keeps every send. if a send was clicked, click_id will have a value. if not, it will be Null.
- i useed function notna() to turn that into True or False.
- only S1 was clicked. S2 and S3 were not.

In [ ]:
# D-Q2: left merge + boolean flag - did each send get clicked?
# left merge keeps all sends, then notna() makes True/False
d_q2 = campaign_sends.merge(clicks, on='send_id', how='left')
d_q2['clicked'] = d_q2['click_id'].notna()
d_q2 = d_q2[['send_id', 'clicked']]
d_q2

Q3: Show all customers and any campaigns they recieved
-join used: left merge 
- i want all customers even if they never got a campaign.
- customers is the left dataframe so eveyone shows up so c@email.com has no sends so campaign_id is Nan for her
- did not use inner merge here becuase that would drop c@email.com.

In [ ]:
# D-Q3: left merge - all customers, with campaigns if any
# c@email.com gets NaN becuase she never recieved a campaign
d_q3 = customers_d.merge(campaign_sends, on='customer_id', how='left')[['email', 'campaign_id']]
d_q3

Q4: Find campaign sends that were never clicked
- join used: left merge + filter Nan anti-join
- i want sends with no clicks. so i left merge sends to clicks then filter where clickid is Nan.
- S2 and S3 were never clicked so they show up and S1 was clicked so it gets filter out

In [ ]:
# D-Q4: anti-join - sends that were never clicked
# left merge then filter where click_id is NaN
merged = campaign_sends.merge(clicks, on='send_id', how='left')
d_q4 = merged[merged['click_id'].isna()][['send_id']]
d_q4

Q5: Find customers who have never recieved a campaign
- join used: left merge + filter NaN with anti-join pattern
- anti-join patern again. left merge customers to sends, then keep only customers where send_id is Nam.
- only c@email.com never got any campaign

In [ ]:
# D-Q5: anti-join - customers who never recieved a campaign
# left merge then filter where send_id is NaN
merged = customers_d.merge(campaign_sends, on='customer_id', how='left')
d_q5 = merged[merged['send_id'].isna()][['email']]
d_q5